# IPC Protocol (020)
Demonstrate the length-prefixed JSON protocol and request handling.

In [11]:
%load_ext autoreload
%autoreload 2
import os
import tempfile
from pathlib import Path

from ciphercache.daemon.state import DaemonConfig, DaemonState
from ciphercache.ipc.handler import handle_request
from ciphercache.ttl import parse_ttl

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
data_dir = Path(tempfile.mkdtemp(prefix="ciphercache-demo-"))
os.chmod(data_dir, 0o700)
config = DaemonConfig(data_dir=data_dir)
state = DaemonState(config=config)
state.unlock(parse_ttl("1h"))
state.secrets["default"] = {"service/api": {"api_key": "demo"}}
ticket_path = state.issue_ticket("demo_client")
ticket = ticket_path.read_text(encoding="utf-8").strip()
ticket

'MAbQyxtc370gmbhhqtkXO8XHjS0bFfJnq0pv3pT052c'

In [13]:
request = {
    "version": "v0",
    "id": "demo-1",
    "type": "request",
    "op": "get_secret",
    "payload": {
        "ticket": ticket,
        "secret_name": "service/api",
        "store": "default",
    },
}
handle_request(state, request)

{'version': 'v0',
 'id': 'demo-1',
 'type': 'response',
 'op': 'get_secret',
 'payload': {'secret': {'api_key': 'demo'}}}